In [1]:
import math

In [65]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), \
            "only supporting int/float powers for now"

        out = Value(
            self.data ** other,
            (self,),
            f'**{other}'
        )

        def _backward():
            self.grad += (
                other *
                self.data ** (other - 1) *
                out.grad
            )

        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def tanh(self):
        n = self.data
        t = (
            math.exp(2*n) - 1
        ) / (
            math.exp(2*n) + 1
        )

        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

In [66]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label = 'f')
L = d * f; L.label = 'L'
L

Value(data=-8.0)

In [67]:
def lol():
    
    h = 0.001
    
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a * b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label = 'f')
    L = d * f; L.label = 'L'
    L1 = L.data
    
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    c.data += h
    e = a * b; e.label = 'e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label = 'f')
    L = d * f; L.label = 'L'
    L2 = L.data
    
    print((L2 - L1)/h)
    
lol()

-1.9999999999988916


In [68]:
# inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# bias of the neuron
b = Value(6.8813735870195432, label='b')

# x1*w1 + x2*w2 + b
x1w1 = x1 * w1
x1w1.label = 'x1*w1'

x2w2 = x2 * w2
x2w2.label = 'x2*w2'

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1*w1 + x2*w2'

n = x1w1x2w2 + b
n.label = 'n'

o = n.tanh()
o.label = 'o'

In [69]:
o.backward()

In [122]:
import random
class Neuron:
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    
class Layer:
    
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(sz[i], sz[i+1])
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        
    
x = [2, 3, 4]
n = MLP(3, [4, 4, 1])
n(x)
n.parameters()

[Value(data=0.038465930255425995),
 Value(data=-0.9509331149689231),
 Value(data=0.057306503137547704),
 Value(data=-0.5777399883168686),
 Value(data=-0.008427767803359743),
 Value(data=0.2648147670786405),
 Value(data=0.5003465266442346),
 Value(data=-0.3674179546678624),
 Value(data=0.12965604626780158),
 Value(data=0.14621062731596757),
 Value(data=-0.22038917417500015),
 Value(data=-0.36295716147949375),
 Value(data=0.4795837623338026),
 Value(data=0.5542052899028211),
 Value(data=0.4041892746617095),
 Value(data=-0.8442525329078601),
 Value(data=-0.36742438078187556),
 Value(data=0.5368467397429877),
 Value(data=-0.9199332570118577),
 Value(data=-0.291913276763033),
 Value(data=0.10270910589521698),
 Value(data=0.6823323430442192),
 Value(data=0.02713680313034983),
 Value(data=0.27103908758005857),
 Value(data=-0.4309276946247229),
 Value(data=-0.9700110862001003),
 Value(data=-0.5247575782520868),
 Value(data=0.950661694560816),
 Value(data=-0.7596733282339794),
 Value(data=-0.56

In [123]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

ypred = [n(x) for x in xs]
ypred

[Value(data=0.4770411044993939),
 Value(data=0.725792142644001),
 Value(data=0.30944559920251374),
 Value(data=0.2971950895717175)]

In [125]:

for k in range(20):

    ypred = [n(x) for x in xs]
    loss = sum((ygt - yout)**2 for ygt, yout in zip(ys, ypred))
    
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()
    
    for p in n.parameters():
        p.data += -0.01 * p.grad
        
    print(k, loss.data)
    

0 2.5472463823492286
1 2.1046402063994476
2 1.7864558446829635
3 1.5465681651284453
4 1.3552830476587208
5 1.1988260941499478
6 1.0689841930769044
7 0.9600469954853728
8 0.867785501364879
9 0.7889789301511152
10 0.721135480617211
11 0.6623053320695951
12 0.6109479267296938
13 0.5658351510349309
14 0.5259796322806254
15 0.49058108589770233
16 0.4589858241185245
17 0.43065595246953214
18 0.4051457526662021
19 0.38208343517136534


In [126]:
ypred

[Value(data=0.7572564154878189),
 Value(data=-0.747434561747722),
 Value(data=-0.657135944969947),
 Value(data=0.6234181011807306)]